# Synthetic Hourly Sin/Cos Data Generator

Генератор минутных данных за период с `2020-02-01` до `2026-02-01` включительно.
Для каждой минуты считается `cos(2πt/T)` и `sin(2πt/T)`, где:
- `t = hour + minute / 60`
- `T = 24`
- `weekday = timestamp.weekday()` (0 = понедельник, 6 = воскресенье)
- `t_week = weekday + hour / 24`
- `cos_weekday = cos(2π * t_week / 7)`
- `sin_weekday = sin(2π * t_week / 7)`

Данные сохраняются в отдельные parquet-файлы по дням, чтобы их можно было удобно объединить по дате или по `timestamp` с другими дневными наборами, например `klines_ada_restored.parquet`.


In [8]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [7]:
# Параметры генерации
START_DATE = "2020-02-01"
END_DATE = "2026-02-01"
OUTPUT_DIR = Path("synthetic_hourly")
T = 24

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("output dir:", OUTPUT_DIR.resolve())

output dir: C:\projects\binance-dowloader-3.0\synthetic_hourly


In [ ]:
def build_daily_synthetic_day(date: str, output_dir: Path) -> Path:
    """Собирает минутные данные для одного дня и сохраняет их в parquet."""
    date_start = pd.Timestamp(date).tz_localize("UTC")
    date_end = date_start + pd.Timedelta(days=1) - pd.Timedelta(minutes=1)

    timestamps = pd.date_range(
        start=date_start,
        end=date_end,
        freq="min",
        tz="UTC",
        inclusive="both",
    )

    df = pd.DataFrame({
        "timestamp": timestamps,
        "hour": timestamps.hour,
        "minute": timestamps.minute,
        "weekday": timestamps.weekday,
    })

    df["t"] = df["hour"] + df["minute"] / 60
    df["cos_hour"] = np.cos(2 * np.pi * df["t"] / T)
    df["sin_hour"] = np.sin(2 * np.pi * df["t"] / T)
    df["t_week"] = df["weekday"] + df["hour"] / 24
    df["cos_weekday"] = np.cos(2 * np.pi * df["t_week"] / 7)
    df["sin_weekday"] = np.sin(2 * np.pi * df["t_week"] / 7)
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    output_path = output_dir / f"synthetic_hourly_{date}.parquet"
    df.to_parquet(output_path, index=False)
    return output_path


# Пример: создаем файл только для одного дня

example_date = "2020-02-01"
example_path = build_daily_synthetic_day(example_date, OUTPUT_DIR)
print("Saved example file:", example_path)

Saved example file: synthetic_hourly\synthetic_hourly_2020-02-01.parquet


In [6]:
# Проверяем содержимое созданного файла
example_df = pd.read_parquet(example_path)
print(example_df.head(8))
print(example_df.tail(3))
print("rows:", len(example_df))
print("timestamp range:", example_df["timestamp"].min(), "-", example_df["timestamp"].max())
print("unique hours:", sorted(example_df["hour"].unique()))

                  timestamp  hour  minute         t  cos_hour  sin_hour
0 2020-02-01 00:00:00+00:00     0       0  0.000000  1.000000  0.000000
1 2020-02-01 00:01:00+00:00     0       1  0.016667  0.999990  0.004363
2 2020-02-01 00:02:00+00:00     0       2  0.033333  0.999962  0.008727
3 2020-02-01 00:03:00+00:00     0       3  0.050000  0.999914  0.013090
4 2020-02-01 00:04:00+00:00     0       4  0.066667  0.999848  0.017452
5 2020-02-01 00:05:00+00:00     0       5  0.083333  0.999762  0.021815
6 2020-02-01 00:06:00+00:00     0       6  0.100000  0.999657  0.026177
7 2020-02-01 00:07:00+00:00     0       7  0.116667  0.999534  0.030539
                     timestamp  hour  minute          t  cos_hour  sin_hour
1437 2020-02-01 23:57:00+00:00    23      57  23.950000  0.999914 -0.013090
1438 2020-02-01 23:58:00+00:00    23      58  23.966667  0.999962 -0.008727
1439 2020-02-01 23:59:00+00:00    23      59  23.983333  0.999990 -0.004363
rows: 1440
timestamp range: 2020-02-01 00:00:00+

In [11]:
# Генерация файлов для первых 7 дней
# Оставляем только первые 7 дат для быстрой проверки.

all_dates = pd.date_range(start=pd.Timestamp(START_DATE), periods=7, freq="D")

for date in all_dates.strftime("%Y-%m-%d"):
    path = OUTPUT_DIR / f"synthetic_hourly_{date}.parquet"
    if not path.exists():
        build_daily_synthetic_day(date, OUTPUT_DIR)
    print("Saved:", path)

print("Completed generation for", len(all_dates), "days")

Saved: synthetic_hourly\synthetic_hourly_2020-02-01.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-02.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-03.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-04.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-05.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-06.parquet
Saved: synthetic_hourly\synthetic_hourly_2020-02-07.parquet
Completed generation for 7 days


In [12]:
# Объединяем все 7 файлов в один DataFrame
merged_frames = []
for date in all_dates.strftime("%Y-%m-%d"):
    file_path = OUTPUT_DIR / f"synthetic_hourly_{date}.parquet"
    merged_frames.append(pd.read_parquet(file_path))

merged_df = pd.concat(merged_frames, ignore_index=True)
print("Merged rows:", len(merged_df))
print("Unique dates:", merged_df["timestamp"].dt.date.nunique())
print("Timestamp range:", merged_df["timestamp"].min(), "-", merged_df["timestamp"].max())

merged_output = OUTPUT_DIR / "synthetic_hourly_first_7_days.parquet"
merged_df.to_parquet(merged_output, index=False)
print("Saved merged file:", merged_output)
merged_df.head()

Merged rows: 10080
Unique dates: 7
Timestamp range: 2020-02-01 00:00:00+00:00 - 2020-02-07 23:59:00+00:00
Saved merged file: synthetic_hourly\synthetic_hourly_first_7_days.parquet


,timestamp,hour,minute,weekday,t,cos_hour,sin_hour,cos_weekday,sin_weekday
0,2020-02-01 00:00:00+00:00,0,0,5,0.000000,1.000000,0.000000,-0.222521,-0.974928
1,2020-02-01 00:01:00+00:00,0,1,5,0.016667,0.999990,0.004363,-0.222521,-0.974928
2,2020-02-01 00:02:00+00:00,0,2,5,0.033333,0.999962,0.008727,-0.222521,-0.974928
3,2020-02-01 00:03:00+00:00,0,3,5,0.050000,0.999914,0.013090,-0.222521,-0.974928
4,2020-02-01 00:04:00+00:00,0,4,5,0.066667,0.999848,0.017452,-0.222521,-0.974928


## Как объединять с другими данными

Если нужно объединить с набором `klines_ada_restored.parquet`, можно сделать это по столбцу `timestamp`.

Пример:

```python
base_df = pd.read_parquet("klines_ada_restored.parquet")
synthetic_df = pd.read_parquet("synthetic_hourly/synthetic_hourly_2020-02-01.parquet")
merged = base_df.merge(synthetic_df, on="timestamp", how="left")
```

Если часы уже присутствуют в `klines_ada_restored.parquet`, синус/косинус будут добавлены к каждой минуте.